# Upper Confidence Bound (UCB) based agents

> Agents utelizing the UCB based approach for Dynamic pricing and learning problems from https://doi.org/10.48550/arXiv.1604.07463

In [ ]:
#| default_exp agents.dynamic_pricing.UCB

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import os
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.dynamic_pricing.mushroom_rl import PricingMushroomBaseAgent
from mushroom_rl.core import Agent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.envs.actionprocessors import ClipAction
from scipy.optimize import minimize

In [ ]:
#| export
class UCBPolicy:
    def __init__(self,
                 lam: float,
                 reg: float,
                 environment_info: MDPInfo,
                 obsprocessors=None,
                 actionprocessors=None,
                 agent_name=None,
                 ex_prices=None,
                 alpha=None,
                 beta=None,
                 price_function=None,
                 g=None):

        if alpha is None:
            alpha = np.zeros(environment_info.observation_space['features'].shape[0])
        if beta is None:
            beta = np.zeros(environment_info.observation_space['features'].shape[0])
        if isinstance(ex_prices, list):
            ex_prices = np.array(ex_prices)
        assert ex_prices.shape[0] >= 2

        self.environment_info = environment_info
        self.ex_prices = ex_prices
        self.alpha = alpha
        self.beta = beta
        self.actionprocessors = actionprocessors or []
        self.obsprocessors = obsprocessors or []
        self.price_function = price_function
        self.g = g
        self.lam = lam
        self.reg = reg
        self.t = 0
        self.X = np.empty((0, environment_info.observation_space['features'].shape[0] * 2))
        self.Y = np.empty((0, 1))
        self.mode = "train"
        self.actionprocessors.append(ClipAction(environment_info.action_space.low, environment_info.action_space.high))

    def draw_action(self, observation):
        x = observation['features']
        if self.t in [0, 1]:
            price = self.ex_prices[self.t]
        else:
            M = self.compute_uncertainty_M(x)
            samples = self.sample_from_confidence_region(np.concatenate([self.alpha, self.beta]), M)
            alpha, beta = self.max_rev(samples, x)
            price = self.price_function(x, alpha, beta)
        for processor in self.actionprocessors:
            price = processor(price)
        return np.array(price, dtype=np.float32)

    def fit(self, X, Y, action):
        self.t += 1
        Z = np.concatenate([X, X * action])
        self.X = np.vstack([self.X, Z])
        self.Y = np.vstack([self.Y, Y])
        self.parameter_update()

    def parameter_update(self):
        if self.X.shape[0] < 2:
            return
        def loss(theta):
            preds = self.g.g(self.X @ theta)
            errors = preds - self.Y.flatten()
            weights = 1 / self.g.v(preds)
            return np.sum((errors**2) * weights) + self.reg * np.linalg.norm(theta)**2

        theta0 = np.concatenate([self.alpha, self.beta])
        res = minimize(loss, theta0, method='L-BFGS-B')
        if res.success:
            theta_hat = res.x
            d = self.environment_info.observation_space['features'].shape[0]
            self.alpha = theta_hat[:d]
            self.beta = theta_hat[d:]

    def sample_design_matrix(self):
        d = self.environment_info.observation_space['features'].shape[0]
        I = self.lam * np.identity(2 * d)
        if self.X.shape[0] == 0:
            return I
        return I + self.X.T @ self.X

    def compute_uncertainty_M(self, x_t):
        M = self.sample_design_matrix()
        d = x_t.shape[0]
        block_matrix = np.block([
            [x_t, np.zeros_like(x_t)],
            [np.zeros_like(x_t), x_t]
        ])
        return np.linalg.inv(block_matrix @ np.linalg.inv(M) @ block_matrix.T)

    def sample_from_confidence_region(self, theta_hat, M, N=50):
        L = np.linalg.cholesky(np.linalg.inv(M))
        u = np.random.randn(len(theta_hat), N)
        u /= np.linalg.norm(u, axis=0)
        return (theta_hat[:, np.newaxis] + (1 / self.environment_info.observation_space['features'].shape[0]) * (L @ u)).T

    def max_rev(self, samples, x):
        max_val = -np.inf
        best_alpha, best_beta = None, None
        for theta in samples:
            alpha = theta[:x.shape[0]]
            beta = theta[x.shape[0]:]
            price = self.price_function(x, alpha, beta)
            rev = price * self.g.g(np.dot(x, alpha) + price * np.dot(x, beta))
            if rev > max_val:
                max_val = rev
                best_alpha, best_beta = alpha, beta
        return best_alpha, best_beta

    def update_task(self, env):
        self.environment_info = env.mdp_info
        d = self.environment_info.observation_space['features'].shape[0]
        self.X = np.empty((0, 2 * d))
        self.Y = np.empty((0, 1))
        self.actionprocessors[-1] = ClipAction(self.environment_info.action_space.low, self.environment_info.action_space.high)
        self.t = 0

    def reset(self):
        pass


In [ ]:
#| export
class UCBCoreAgent(Agent):

    """
    Base class for UCB agents.
    """

    def __init__(self,
                 lam: float,
                 reg: float,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = [],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        
        policy = UCBPolicy(lam=lam, reg=reg, environment_info=environment_info, obsprocessors=obsprocessors, actionprocessors=actionprocessors, ex_prices=ex_prices, alpha=alpha, beta=beta, price_function=price_function, g=g)
        self.agent_name = agent_name
        super().__init__(environment_info, policy)
        
    def fit(self, dataset, **kwargs):
        X = dataset[0][0]['features']
        Y = kwargs["demand"][0]
        action = dataset[0][1]
        self.policy.fit(X, Y, action)
        
    def update_task(self, env):
        self.policy.update_task(env)

In [ ]:
#| export
class UCBAgent(PricingMushroomBaseAgent):
    """
    Wrapper class for UCBCoreAgent to interact with MushroomRL.
    """
    def __init__(self,
                 lam: float,
                 reg: float,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] =[],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        self.agent = UCBCoreAgent(lam=lam, reg=reg, environment_info=environment_info,
                                  obsprocessors=obsprocessors, 
                                  actionprocessors=actionprocessors, 
                                  agent_name=agent_name, 
                                  ex_prices=ex_prices, 
                                  alpha=alpha, 
                                  beta=beta, 
                                  price_function=price_function, 
                                  g=g)
        super().__init__(environment_info=environment_info, obsprocessors=obsprocessors, agent_name=agent_name)
    def update_task(self, env: object):
        """ Update the environment specific parameters of the agent """
        self.agent.update_task(env)